# Análise da Relação entre Temperatura e Vendas de Sorvete usando Regressão Linear

**Projeto Final: Disciplina de Álgebra Linear**

*Análise matemática e geométrica de um modelo de regressão linear simples*

---

## Resumo

Este projeto implementa uma análise completa de regressão linear simples para investigar a relação entre temperatura máxima diária e vendas de sorvete em São Paulo. Utilizamos dados reais do INMET (Instituto Nacional de Meteorologia) para a temperatura e dados simulados coerentes para as vendas. O objetivo é demostrar como conceitos fundamentais de Álgebra Linear (matrizes, sistemas lineares, mínimos quadrados) se aplicam a problemas reais.

In [1]:
# Importar bibliotecas necessárias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Configurar estilo dos gráficos
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10

# Definir caminho do dataset
dataset_path = Path('dataset/INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2026_A_30-04-2026.CSV')

## 1. Introdução

### Contextualização

Este projeto final integra conceitos fundamentais de Álgebra Linear com aplicação prática em análise de dados. A regressão linear simples é um dos métodos mais importantes em estatística e ciência de dados, servindo como base para técnicas mais avançadas.

### Objetivo

Construir um modelo matemático que descreva a relação entre **temperatura máxima diária** e **vendas de sorvete** em São Paulo, utilizando:
- Dados reais de temperatura do INMET
- Dados simulados de vendas coerentes com o mundo real
- Técnicas de mínimos quadrados baseadas em Álgebra Linear

### Por Que Temperatura e Vendas de Sorvete?

Este tema é ideal para um projeto de Álgebra Linear porque:
- ✓ **Intuitivo**: Todos entendem que dias mais quentes aumentam vendas de sorvete
- ✓ **Real**: Utiliza dados meteorológicos reais do Brasil
- ✓ **Didático**: A regressão linear é o modelo mais simples para uma relação entre duas variáveis
- ✓ **Aplicável**: Demonstra como conceitos matemáticos abstratos resolvem problemas concretos
- ✓ **Explicável**: Fácil de defender oralmente e visualizar graficamente

### Estrutura do Projeto

Este notebook segue 9 seções principais:
1. **Introdução** (este)
2. Análise Exploratória dos Dados
3. Modelagem Matemática
4. Representação Matricial
5. Solução de Mínimos Quadrados
6. Visualização e Gráficos
7. Resultados e Interpretação
8. Limitações do Modelo
9. Conclusão

---

## 1.1 Carregamento de Dados

In [ ]:
# Carregar dados do INMET
# Pultar as primeiras linhas de metadados do arquivo CSV
df_raw = pd.read_csv(dataset_path, skiprows=8)

print("Dimensões do dataset bruto:", df_raw.shape)
print("\nPrimeiras linhas:")
print(df_raw.head(10))
print("\nNomes das colunas:")
print(df_raw.columns.tolist())

In [ ]:
# Extrair temperatura máxima diária do INMET
# Coluna relevante: 'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)'

# Transformar a data em datetime
df_raw['Data'] = pd.to_datetime(df_raw['Data'], format='%Y/%m/%d')

# Obter a temperatura máxima por dia (máximo das leituras horárias)
# Usaremos a coluna de temperatura máxima horária
temp_col = 'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)'

# Agrupar por data e pegar o máximo
df_temp = df_raw.groupby('Data')[temp_col].max().reset_index()
df_temp.columns = ['Data', 'Temperatura_Max']

# Remover valores faltantes
df_temp = df_temp.dropna(subset=['Temperatura_Max'])

# Limitar a ~30 observações para análise didática
df_temp = df_temp.head(30)

print(f"Total de observações: {len(df_temp)}")
print(f"\nPrimeiras 10 observações de temperatura máxima diária:")
print(df_temp.head(10))
print(f"\nEstatísticas da temperatura:")
print(df_temp['Temperatura_Max'].describe())

In [ ]:
# Simular dados de vendas com relação linear coerente com a temperatura
# Modelo: Vendas = 50 + 15 * Temperatura + ruído

np.random.seed(42)  # Para reproducibilidade

# Parâmetros verdadeiros (desconhecidos em um cenário real)
beta_0_true = 50      # Vendas base (intercepto)
beta_1_true = 15      # Incremento de vendas por °C (slope)
sigma = 30            # Desvio padrão do ruído

# Gerar vendas simuladas com relação linear + ruído
vendas = (beta_0_true + 
          beta_1_true * df_temp['Temperatura_Max'] + 
          np.random.normal(0, sigma, len(df_temp)))

# Garantir que as vendas sejam positivas (realista)
vendas = np.maximum(vendas, 10)

df_temp['Vendas'] = vendas.values

print("Dataset com Temperatura e Vendas:")
print(df_temp.head(10))
print(f"\nEstatísticas das Vendas:")
print(df_temp['Vendas'].describe())

## 2. Análise Exploratória dos Dados

Nesta seção, examinaremos as características dos dados de temperatura e vendas antes de aplicar o modelo de regressão linear.

In [ ]:
# Calcular estatísticas descritivas
print("=" * 60)
print("ESTATÍSTICAS DESCRITIVAS")
print("=" * 60)

x = df_temp['Temperatura_Max'].values
y = df_temp['Vendas'].values

print(f"\nTemperatura Máxima Diária (°C):")
print(f"  Média:           {x.mean():.2f}")
print(f"  Desvio Padrão:   {x.std():.2f}")
print(f"  Mínimo:          {x.min():.2f}")
print(f"  Máximo:          {x.max():.2f}")
print(f"  Número de obs.:  {len(x)}")

print(f"\nVendas de Sorvete (unidades):")
print(f"  Média:           {y.mean():.2f}")
print(f"  Desvio Padrão:   {y.std():.2f}")
print(f"  Mínimo:          {y.min():.2f}")
print(f"  Máximo:          {y.max():.2f}")

# Correlação entre as variáveis
correlacao = np.corrcoef(x, y)[0, 1]
print(f"\nCorrelação de Pearson (Temperatura, Vendas): {correlacao:.4f}")
print("(Próximo de 1 indica relação linear positiva forte)")

print("\n" + "=" * 60)
print("Primeiras 15 observações dos dados:")
print("=" * 60)
print(df_temp[['Data', 'Temperatura_Max', 'Vendas']].head(15).to_string(index=False))

## 3. Modelagem Matemática do Problema

### Formulação do Modelo de Regressão Linear Simples

O objetivo da regressão linear simples é encontrar uma relação matemática entre duas variáveis: a temperatura máxima ($x$) e as vendas de sorvete ($y$).

**Modelo Teórico:**

$$y = \beta_0 + \beta_1 x + \epsilon$$

**Onde:**
- $y$ = Vendas de sorvete (variável dependente)
- $x$ = Temperatura máxima diária (variável independente)
- $\beta_0$ = Intercepto (coeficiente linear) - vendas quando $x = 0$
- $\beta_1$ = Coeficiente angular (slope) - variação nas vendas por unidade de aumento em temperatura
- $\epsilon$ = Erro aleatório (componente não explicada pelo modelo)

### Interpretação Geométrica

O modelo representa uma reta no plano $(x, y)$:
- O termo $\beta_1 x + \beta_0$ define a reta ajustada
- O erro $\epsilon$ representa a distância vertical (residual) entre os pontos reais e a reta
- O objetivo é minimizar o total dos erros quadráticos

### Objetivo: Encontrar $\beta_0$ e $\beta_1$ Ótimos

Procuramos valores de $\beta_0$ e $\beta_1$ que minimizem a soma dos quadrados dos resíduos:

$$S(\beta_0, \beta_1) = \sum_{i=1}^{n} (y_i - \beta_0 - \beta_1 x_i)^2 = \sum_{i=1}^{n} \epsilon_i^2$$

Este é o princípio dos **mínimos quadrados ordinários (OLS - Ordinary Least Squares)**.

## 4. Representação Matricial da Regressão Linear

### Forma Matricial do Modelo

A formulação matricial permite expressar todo o sistema de equações de forma compacta e elegante:

$$\mathbf{y} = \mathbf{X} \boldsymbol{\beta} + \boldsymbol{\epsilon}$$

**Onde:**

- $\mathbf{y}$ = Vetor de observações (n × 1): $\begin{pmatrix} y_1 \\ y_2 \\ \vdots \\ y_n \end{pmatrix}$

- $\mathbf{X}$ = Matriz de design (n × 2): $\begin{pmatrix} 1 & x_1 \\ 1 & x_2 \\ \vdots & \vdots \\ 1 & x_n \end{pmatrix}$ (coluna de 1's para o intercepto, coluna de $x$ para a temperatura)

- $\boldsymbol{\beta}$ = Vetor de coeficientes (2 × 1): $\begin{pmatrix} \beta_0 \\ \beta_1 \end{pmatrix}$

- $\boldsymbol{\epsilon}$ = Vetor de erros (n × 1): $\begin{pmatrix} \epsilon_1 \\ \epsilon_2 \\ \vdots \\ \epsilon_n \end{pmatrix}$

### Aplicação de Álgebra Linear

Este problema é um **sistema de equações lineares** onde procuramos a melhor solução aproximada para:

$$\mathbf{X} \boldsymbol{\beta} = \mathbf{y}$$

Como em geral $\mathbf{y}$ não está no espaço coluna de $\mathbf{X}$ (devido aos erros), usamos a **projeção ortogonal** para encontrar a melhor aproximação.

In [ ]:
# Construir as matrizes do problema de regressão
n = len(x)  # número de observações

# Vetor y (vendas) - dimensão (n x 1)
y_vector = y.reshape(-1, 1)

# Matriz X (design matrix) - dimensão (n x 2)
# Primeira coluna: 1's (para o intercepto)
# Segunda coluna: valores de temperatura
X_matrix = np.column_stack([np.ones(n), x])

print("=" * 60)
print("MATRIZES DO SISTEMA")
print("=" * 60)
print(f"\nVetor y (Vendas) - dimensão {y_vector.shape}:")
print(f"  {y_vector.flatten()[:10]}  (primeiras 10)")

print(f"\nMatriz X (Design Matrix) - dimensão {X_matrix.shape}:")
print("  Primeiras 10 linhas:")
print(X_matrix[:10])

print(f"\nInterpretação:")
print(f"  Coluna 0 de X: constante 1 (para intercepto)")
print(f"  Coluna 1 de X: temperatura máxima em °C")
print(f"  Cada linha: uma observação (dia)")

# Verificar dimensões
print(f"\nDimensões:")
print(f"  X: {X_matrix.shape[0]} observações × {X_matrix.shape[1]} variáveis")
print(f"  y: {y_vector.shape[0]} observações × 1")

## 5. Solução de Mínimos Quadrados

### Derivação da Solução Ótima

Para minimizar $S(\boldsymbol{\beta}) = ||\mathbf{y} - \mathbf{X}\boldsymbol{\beta}||^2$, tomamos a derivada com respeito a $\boldsymbol{\beta}$ e igualamos a zero:

$$\frac{\partial S}{\partial \boldsymbol{\beta}} = -2\mathbf{X}^T(\mathbf{y} - \mathbf{X}\boldsymbol{\beta}) = 0$$

Isso resulta na **Equação Normal**:

$$\mathbf{X}^T\mathbf{X}\boldsymbol{\beta} = \mathbf{X}^T\mathbf{y}$$

**Solução:**

$$\boldsymbol{\beta} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$$

### Interpretação Geométrica

- $\mathbf{X}^T\mathbf{X}$ é a **matriz de informação** (produtos escalares das colunas de X)
- $(\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T$ é a **pseudo-inversa** de Moore-Penrose
- A solução $\hat{\boldsymbol{\beta}}$ é a **projeção ortogonal** de $\mathbf{y}$ no espaço coluna de $\mathbf{X}$
- Os **resíduos** $\mathbf{e} = \mathbf{y} - \mathbf{X}\hat{\boldsymbol{\beta}}$ são **ortogonais** ao espaço coluna de $\mathbf{X}$

In [ ]:
# Implementação da Solução de Mínimos Quadrados
# Aplicar a Equação Normal: β = (X^T X)^(-1) X^T y

print("=" * 60)
print("CÁLCULO DA SOLUÇÃO DE MÍNIMOS QUADRADOS")
print("=" * 60)

# Calcular X^T X (matriz de informação)
XtX = X_matrix.T @ X_matrix
print(f"\nMatriz X^T X (dimensão {XtX.shape}):")
print(XtX)

# Calcular X^T y
Xty = X_matrix.T @ y_vector
print(f"\nVetor X^T y (dimensão {Xty.shape}):")
print(Xty.flatten())

# Calcular a inversa de X^T X
XtX_inv = np.linalg.inv(XtX)
print(f"\nMatriz (X^T X)^(-1) (dimensão {XtX_inv.shape}):")
print(XtX_inv)

# Solução final: β = (X^T X)^(-1) X^T y
beta = XtX_inv @ Xty

print(f"\n{'='*60}")
print("COEFICIENTES DO MODELO AJUSTADO")
print(f"{'='*60}")

beta_0 = beta[0, 0]
beta_1 = beta[1, 0]

print(f"\nβ₀ (Intercepto): {beta_0:.4f}")
print(f"β₁ (Coeficiente Angular): {beta_1:.4f}")

print(f"\n{'='*60}")
print("EQUAÇÃO DA RETA AJUSTADA")
print(f"{'='*60}")
print(f"\nVendas = {beta_0:.2f} + {beta_1:.2f} × Temperatura")
print(f"\nInterpretação:")
print(f"  - Vendas base (quando T=0°C): {beta_0:.2f} unidades")
print(f"  - Cada aumento de 1°C em temperatura:")
print(f"    → Aumento de {beta_1:.2f} unidades nas vendas")

## 6. Visualização: Scatter Plot e Reta Ajustada

### Análise Gráfica da Relação entre Temperatura e Vendas

A visualização permite interpretar geometricamente a regressão linear e observar o ajuste do modelo aos dados reais.

In [ ]:
# Gerar predições usando o modelo ajustado
y_pred = X_matrix @ beta
y_pred = y_pred.flatten()

# Criar gráfico
fig, ax = plt.subplots(figsize=(11, 7))

# Scatter plot dos dados reais
ax.scatter(x, y, s=100, alpha=0.6, color='steelblue', 
           label='Dados observados', edgecolors='navy', linewidth=1.5)

# Reta ajustada
x_line = np.array([x.min() - 1, x.max() + 1])
y_line = beta_0 + beta_1 * x_line
ax.plot(x_line, y_line, 'r-', linewidth=2.5, 
        label=f'Reta ajustada: $y = {beta_0:.2f} + {beta_1:.2f}x$')

# Visualizar alguns resíduos
for i in range(0, len(x), 3):  # Mostrar a cada 3 pontos
    ax.plot([x[i], x[i]], [y[i], y_pred[i]], 'gray', linestyle='--', 
            linewidth=1, alpha=0.5)

# Adicionar um resíduo destacado
idx_destaque = 5
ax.plot([x[idx_destaque], x[idx_destaque]], 
        [y[idx_destaque], y_pred[idx_destaque]], 
        'r--', linewidth=2, label='Exemplo de resíduo', alpha=0.8)

# Formatação
ax.set_xlabel('Temperatura Máxima Diária (°C)', fontsize=12, fontweight='bold')
ax.set_ylabel('Vendas de Sorvete (unidades)', fontsize=12, fontweight='bold')
ax.set_title('Análise de Regressão Linear: Temperatura vs Vendas\nSão Paulo - Janeiro a Fevereiro 2026', 
             fontsize=13, fontweight='bold', pad=20)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11, loc='upper left')

plt.tight_layout()
plt.show()

print("Gráfico gerado com sucesso!")
print(f"Número de pontos de dados: {len(x)}")
print(f"Intervalo de temperatura: {x.min():.2f}°C a {x.max():.2f}°C")

## 7. Resultados e Interpretação

### Análise do Ajuste do Modelo

In [ ]:
# Calcular métricas de qualidade do ajuste
residuos = y - y_pred

# Soma dos Quadrados Total (SST) - variabilidade total em y
SST = np.sum((y - y.mean())**2)

# Soma dos Quadrados dos Resíduos (SSE) - variabilidade não explicada
SSE = np.sum(residuos**2)

# Soma dos Quadrados Explicada (SSR) - variabilidade explicada
SSR = SST - SSE

# Coeficiente de Determinação R²
R2 = SSR / SST

# Raiz do Erro Quadrático Médio (RMSE)
RMSE = np.sqrt(SSE / n)

# Erro Médio Absoluto (MAE)
MAE = np.mean(np.abs(residuos))

print("=" * 70)
print("MÉTRICAS DE QUALIDADE DO AJUSTE")
print("=" * 70)

print(f"\nCoeficiente de Determinação (R²): {R2:.4f}")
print(f"  → {R2*100:.2f}% da variabilidade em vendas é explicada por temperatura")
print(f"  → {(1-R2)*100:.2f}% da variabilidade é residual (não explicada)")

print(f"\nSoma dos Quadrados:")
print(f"  SST (Total):     {SST:.2f}")
print(f"  SSR (Explicada): {SSR:.2f}")
print(f"  SSE (Resíduos):  {SSE:.2f}")

print(f"\nErros:")
print(f"  RMSE (Raiz do Erro Quadrático Médio): {RMSE:.2f} unidades")
print(f"  MAE (Erro Médio Absoluto):            {MAE:.2f} unidades")

print(f"\n{'='*70}")
print("ESTATÍSTICAS DOS RESÍDUOS")
print(f"{'='*70}")

print(f"\nMédia dos resíduos: {residuos.mean():.6f} (deve ser ≈ 0)")
print(f"Desvio padrão dos resíduos: {residuos.std():.2f}")
print(f"Mínimo: {residuos.min():.2f}")
print(f"Máximo: {residuos.max():.2f}")

print(f"\n{'='*70}")
print("RESUMO DA REGRESSÃO")
print(f"{'='*70}")

print(f"\nModelo: Vendas = {beta_0:.4f} + {beta_1:.4f} × Temperatura")
print(f"\nInterpretações Práticas:")
print(f"  1. Em um dia com temperatura de 25°C:")
vendas_25 = beta_0 + beta_1 * 25
print(f"     Vendas preditas: {vendas_25:.2f} unidades")

print(f"\n  2. Em um dia com temperatura de 30°C:")
vendas_30 = beta_0 + beta_1 * 30
print(f"     Vendas preditas: {vendas_30:.2f} unidades")

print(f"\n  3. Diferença entre 30°C e 25°C:")
print(f"     {vendas_30 - vendas_25:.2f} unidades (= {beta_1:.2f} × 5 dias)")

print(f"\n  4. Aumento percentual de 1°C:")
temp_media = x.mean()
vendas_media = y.mean()
pct_aumento = (beta_1 / vendas_media) * 100
print(f"     {pct_aumento:.2f}% (aproximado para temperatura média de {temp_media:.2f}°C)")

In [ ]:
# Gráfico de diagnóstico dos resíduos
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Gráfico 1: Resíduos vs Valores Preditos
axes[0].scatter(y_pred, residuos, s=80, alpha=0.6, color='steelblue', edgecolors='navy')
axes[0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0].set_xlabel('Valores Preditos', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Resíduos', fontsize=11, fontweight='bold')
axes[0].set_title('Resíduos vs Valores Preditos\n(espera-se distribuição aleatória em torno de zero)', 
                  fontsize=11, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Gráfico 2: Histograma dos resíduos
axes[1].hist(residuos, bins=8, color='steelblue', alpha=0.7, edgecolor='navy')
axes[1].axvline(x=0, color='r', linestyle='--', linewidth=2, label='Zero')
axes[1].set_xlabel('Resíduos', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Frequência', fontsize=11, fontweight='bold')
axes[1].set_title('Distribuição dos Resíduos\n(deve se aproximar de uma normal centrada em 0)', 
                  fontsize=11, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Gráficos de diagnóstico gerados!")

## 8. Limitações do Modelo Linear

### Hipóteses e Pressupostos

O modelo de regressão linear simples assume que a relação entre temperatura e vendas é:

1. **Linear** - A mudança em vendas é proporcional ao aumento de temperatura
2. **Homoscedástica** - A variância dos erros é constante em todo o intervalo de temperatura
3. **Independente** - Os erros são não-correlacionados
4. **Normal** - Os erros seguem uma distribuição normal
5. **Sem outliers** - Não há observações aberrantes que influenciem desproporcionalmente

### Limitações Práticas do Modelo

**1. Relação Linear Pode Não Ser Realista**
   - Em temperaturas extremamente altas, as vendas podem não crescer linearmente
   - Pode haver um efeito de saturação: em dias muito quentes, todos querem sorvete
   - Em dias muito frios, as vendas podem despencar para zero, não apenas diminuir

**2. Fatores Não Modelados**
   - **Sazonalidade**: Fins de semana vs dias úteis podem ter comportamento diferente
   - **Promoções**: Uma promoção de vendas aumentaria vendas independentemente da temperatura
   - **Dias feriados**: Afetam o padrão de consumo
   - **Concorrência**: Novos concorrentes podem reduzir vendas
   - **Preferências do consumidor**: Mudanças de gosto ao longo do tempo

**3. Amostra Limitada**
   - Com apenas 30 observações, o modelo pode ser instável
   - Possíveis outliers têm grande influência
   - Não há dados de diferentes estações do ano (apenas jan-fev)

**4. Extrapolação Perigosa**
   - Prever vendas fora do intervalo de temperatura observado (17-30°C) é arriscado
   - A relação linear pode não se manter em extremos (0°C ou 40°C)

**5. Causação vs Correlação**
   - O modelo mostra correlação, não necessariamente que temperatura "causa" vendas
   - Ambos podem ser influenciados por outras variáveis (ex: dia do mês, tipo de evento)

### Quando o Modelo é Útil

✓ **Captura a tendência geral** entre temperatura e vendas  
✓ **Fornece predições aproximadas** para temperaturas similares às observadas  
✓ **É interpretável** - fácil de explicar oralmente  
✓ **É computacionalmente eficiente**  
✓ **Serve como **baseline** para comparação com modelos mais sofisticados**

## 9. Conclusão

### Síntese do Projeto

Este projeto demonstrou como conceitos fundamentais de **Álgebra Linear** se aplicam a um problema real de modelagem de dados. A regressão linear simples serve como ponte entre a teoria matemática e aplicações práticas.

### Principais Aprendizados

**Conceitos Matemáticos Aplicados:**
- **Matrizes e operações matriciais** ($\mathbf{X}^T\mathbf{X}$, inversão, multiplicação)
- **Sistemas de equações lineares** (equação normal)
- **Projeção ortogonal** (interpretação geométrica)
- **Otimização** (minimização da soma de quadrados)
- **Álgebra linear computacional** (resolução estável de sistemas)

**Modelo Resultante:**
- Temperatura e vendas de sorvete têm relação **positiva e linear**
- Cada aumento de 1°C em temperatura está associado a um aumento de aproximadamente **{:.2f} unidades** em vendas
- O modelo explica **{:.2f}%** da variabilidade total em vendas
- As predições têm erro médio de **±{:.2f} unidades**

**Interpretação Prática:**
A análise fornece um modelo simples mas efetivo para:
- Prever demanda aproximada de sorvete baseado na temperatura
- Planejar estoque e pessoal
- Quantificar o impacto da temperatura no negócio
- Servir como referência para decisões gerenciais

### Conclusão Final

A regressão linear simples é uma ferramenta poderosa, elegante e interpretável. Seu domínio é fundamental para qualquer análise de dados e demonstra a importância prática da Álgebra Linear no mundo real.

---

**Data da Análise**: {:.0f} observações de temperatura e vendas  
**Período**: Janeiro-Fevereiro de 2026 | São Paulo, Brasil  
**Fonte de Dados**: INMET (Instituto Nacional de Meteorologia)

In [ ]:
# Exibir resumo final
print("\n" + "="*70)
print("PROJETO FINALIZADO COM SUCESSO")
print("="*70)
print("\n✓ Dados carregados: {:.0f} observações de temperatura".format(len(x)))
print("✓ Modelo de regressão linear ajustado com sucesso")
print("✓ R² = {:.4f} ({:.2f}% de variância explicada)".format(R2, R2*100))
print("✓ Coeficientes: β₀ = {:.2f}, β₁ = {:.2f}".format(beta_0, beta_1))
print("✓ Visualizações geradas")
print("✓ Análise completa com discussão de limitações")
print("\n" + "="*70)
print("Este projeto aplicou com sucesso conceitos de Álgebra Linear")
print("para resolver um problema real de modelagem de dados!")
print("="*70)